In [29]:
!pip install chromadb

In [ ]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.2 MB/s eta 0:00:00


In [30]:
import os
import chromadb
import requests
from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from PyPDF2 import PdfReader
from google.colab import userdata
from google import genai

In [31]:
from sentence_transformers import CrossEncoder

In [32]:
OPENAI_API_KEY = userdata.get('openai_key')
GEMINI_API_KEY = userdata.get('gemini_key')

# Knowledge Base:

### Loading previous knowledge base:
-if loading previous knowledge base, just run the following cell   
-if building knowledge base, run all cells in this section

In [33]:
# chroma_client = chromadb.PersistentClient(path='/content/')
# collection = client.create_collection(
#     name="my_collection",
#     embedding_function=OpenAIEmbeddingFunction(
#         model_name="text-embedding-3-small"
#         api_key_env_var=OPENAI_API_KEY
#     )
# )

chroma_client = chromadb.PersistentClient(path='/content/drive/MyDrive/toyRag/chroma_db')
collection = chroma_client.get_or_create_collection(name="snake_collection")

In [ ]:
def get_text_txt_md(file_path: str) -> str:
  with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()
  return text

def get_text_pdf(pdf_path: str) -> str:
    """Extract raw text from a PDF file."""
    try:
        reader = PdfReader(pdf_path)
        return " ".join(page.extract_text() for page in reader.pages if page.extract_text())
    except Exception as e:
        raise RuntimeError(f"Error reading PDF: {e}")

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 25) -> list[str]:
    """Split text into manageable chunks for embeddings."""
    words = text.split()
    return [" ".join(words[i-overlap:i+chunk_size]) for i in range(overlap, len(words), chunk_size - overlap)]

In [ ]:
def chunk_text_

In [ ]:
files_paths = ['/content/Snake_wikiWikipedia.pdf', '/content/cool_math.pdf', '/content/its_nice_that.txt']
chunks = []
text = ""
for file in files_paths:
  if "pdf" in file:
    text = get_text_pdf(file)
  else:
    text = get_text_txt_md(file)
  chunks += chunk_text(text)

collection.upsert(
    documents=chunks,
    ids=[f"id{i}" for i in range(len(chunks))]
)

In [34]:
#test retrieve:
collection.query(
      query_texts=["Who was the original creator?"],
      n_results=4
  )

{'ids': [['id27', 'id23', 'id31', 'id16']],
 'embeddings': None,
 'documents': [['purposes. It was a game with two snakes, controlled by two players both on their own side of the keyboard. So I suggested we test whether we can implement that between two handsets, both of course controlling their snake with their own handset.” Tetris was another game considered for transformation into the mobile sphere, yet there were inevitably some issues with copyright. Snake, on the other hand, was a simple game – one that didn’t take up much space, which displayed nicely and was fully controllable with minimal keys. It also had no issues in terms of copyright and was ready to be built from scratch. “So we decided to go for it.” Snake was conceived in programming language C, just like many other parts of the software in handsets of the generation. The game was “hand-written” line-by-line, and no specific tools or code generators were needed (or available) to do so. While in the testing phase and wor

# Generation:

In [ ]:
rr_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
def rerank(chunks, prompt, k, rr_model){
    return rr_model.rank(query, chunks, return_documents=True, top_k=3)
}

In [ ]:
def call_llm(name: str, model: str, prompt: str, client):
  if name == "gemini":
    try:
      response = client.interactions.create(
          model=model,
          input=prompt
      )
      return response.output_text
    except Exception as e:
        raise RuntimeError(f"Gemini LLM query failed: {e}")



In [45]:
def gen_response(query, client, rr_model, collection, k, kp):
  k_chunks = collection.query(
      query_texts=[query],
      n_results=k
  )
  kp_chunks = rr_model.rank(query, chunks, return_documents=True, top_k=kp);
  kp_chunks = [item["text"] for item in kp_chunks]
  context = "\n".join(kp_chunks)
  prompt = f"{query} Only use the following context to answer this question. Clearly state when the context does not contain the answer: {context}"
  return call_llm("gemini", "gemini-3.6-flash", prompt, client), k_chunks, kp_chunks


In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [47]:
client = genai.Client(api_key=GEMINI_API_KEY)
rr_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
query = "When was the snake game made?"
response, k_chunks, kp_chunks = gen_response(query, client, rr_model, collection, 10, 3)
print(response)
print(kp_chunks)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Based on the provided context, there is no single date given for when "the snake game" was made, as it refers to a genre with multiple versions:

* The first version of the Snake-game genre was a game called **Blockade**, published in **1976**.
* Other early versions titled *Snake* or *Snake Byte* were published in **1982** (such as *Snake Byte* and *Snake* for the BBC Micro).
* The mobile game specifically named **Snake** was released for the Nokia 6110 in **1998**.
["QBasic sample program. In 1992, Rattler Race was released as part of the second Microsoft Entertainment Pack . It adds enemy snakes to the familiar apple-eating gameplay. In 1998, the mobile game Snake was released for Nokia 6110 .[10] The game was popular, and Nokia released a series of reiterations, including Snake II, Snake EX, Snake EX2 , Snake III, Snakes , Snake Xenzia and Snakes Subsonic . As the game graphics and gameplay evolved, it became less popular.[11] In 2002, Snake was made available for download to Pocke

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

{'ids': [['id16', 'id7', 'id20', 'id4', 'id1']], 'embeddings': None, 'documents': [['Griffin Bateson / January 4, 2023 Snake is perhaps one of the most well-known titles in all of gaming history, going all the way back to the days when arcades were the most popular way to play games. This simple game has quite the history, so let’s just hop straight into the history of Snake and how it all began. Origins Of The Snake Game Like so many of the retro games that we have come to know and love here at Coolmath Games, Snake started in the arcade. You know, those places that had tons of coin-operated games for players to play.The History Of Snake The Game Home Categories Search Profile EN Interestingly enough, though, the first version of Snake wasn’t actually called Snake. It was a very similar game called Blockade. Blockade was created by the gaming manufacturer Gremlin, a San Diego-based gaming company. They published the game in 1976, right around when another famous game, Atari Breakout, 

In [39]:
rr_chunks = rr_model.rank(query, chunks, return_documents=True, top_k=3)
print(rr_chunks)

[{'corpus_id': 4, 'score': np.float32(6.584159), 'text': "QBasic sample program. In 1992, Rattler Race was released as part of the second Microsoft Entertainment Pack . It adds enemy snakes to the familiar apple-eating gameplay. In 1998, the mobile game Snake was released for Nokia 6110 .[10] The game was popular, and Nokia released a series of reiterations, including Snake II, Snake EX, Snake EX2 , Snake III, Snakes , Snake Xenzia and Snakes Subsonic . As the game graphics and gameplay evolved, it became less popular.[11] In 2002, Snake was made available for download to Pocket PC through Peter's GameBox.[12] In 2004, TIM made Snake available for download through the Tim Wap Fast system.[13] In March 2013, NimbleBit released Nimble Quest , an action RPG snake game.[14] In 2015, Armanto released aHistory Later games7/10/26, 10:30 AM Snake (video game genre) - Wikipedia https://en.wikipedia.org/wiki/Snake_(video_game_genre) 2/7 spiritual successor to Snake in partnership with Rumilus De